In [4]:
import esa_snappy
from esa_snappy import ProductIO

print("SNAP Python OK")


SNAP Python OK


In [5]:
from sentinelsat import SentinelAPI
import geopandas as gpd
print("EO stack OK")


EO stack OK


In [6]:
import geopandas as gpd

files = [
    "basin_bc.geojson",
    "basin_grid.gpkg"
]

for f in files:
    print("\n==============================")
    print("FILE:", f)

    gdf = gpd.read_file(f)

    print("CRS:", gdf.crs)
    print("Features:", len(gdf))
    print("Geometry type:", gdf.geometry.iloc[0].geom_type)
    print("Is valid:", gdf.is_valid.all())
    print("Bounds:", gdf.total_bounds)

    # якщо CRS не WGS84 — покажемо площу після проєкції
    if gdf.crs is None:
        print("⚠ CRS missing!")
    else:
        gdf_m = gdf.to_crs(3857)
        area_km2 = gdf_m.area.sum() / 1e6
        print("Area (km²):", round(area_km2, 2))



FILE: basin_bc.geojson
CRS: EPSG:4326
Features: 1
Geometry type: MultiPolygon
Is valid: True
Bounds: [24.71705011 47.69400697 25.12350515 48.12574983]
Area (km²): 1569.76

FILE: basin_grid.gpkg
CRS: EPSG:4326
Features: 10
Geometry type: MultiPolygon
Is valid: True
Bounds: [24.71705011 47.69400697 25.12350515 48.12574983]
Area (km²): 1569.76


In [7]:
from sentinelsat import SentinelAPI, geojson_to_wkt
import geopandas as gpd
from datetime import date
import logging

logging.basicConfig(level=logging.INFO)

USER = "nikoriakviktor@gmail.com"
PASS = "456140NiKo&maria"

api = SentinelAPI(USER, PASS, "https://dataspace.copernicus.eu")

print(api.logger)

<Logger sentinelsat.SentinelAPI (INFO)>


In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

CDSE_USER = os.getenv("CDSE_USER")
CDSE_PASS = os.getenv("CDSE_PASS")

print(CDSE_USER)


nikoriakviktor@gmail.com


In [9]:
import geopandas as gpd

gdf = gpd.read_file("basin_bc.geojson").to_crs(4326)
minx, miny, maxx, maxy = gdf.total_bounds


In [10]:
import os
import requests
import geopandas as gpd
from datetime import date

TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
DOWNLOAD = "https://zipper.dataspace.copernicus.eu/odata/v1/Products"


def get_token(user: str, pwd: str) -> str:
    r = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "password",
            "client_id": "cdse-public",
            "username": user,
            "password": pwd,
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["access_token"]


def wkt_bbox_4326(geojson_path: str) -> str:
    """Build WKT POLYGON without space before parens (CDSE requirement)."""
    gdf = gpd.read_file(geojson_path).to_crs(4326)
    minx, miny, maxx, maxy = gdf.total_bounds
    return f"POLYGON(({minx} {miny},{maxx} {miny},{maxx} {maxy},{minx} {maxy},{minx} {miny}))"


token = get_token(CDSE_USER, CDSE_PASS)

wkt = wkt_bbox_4326("basin_bc.geojson")

# Flood period: June 2020 (Bukovyna flood)
start = "2020-06-11"
end = "2020-06-30"

odata_filter = (
    "Collection/Name eq 'SENTINEL-1' and "
    f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt}') and "
    f"ContentDate/Start gt {start}T00:00:00.000Z and "
    f"ContentDate/Start lt {end}T23:59:59.999Z and "
    "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
    "and att/OData.CSC.StringAttribute/Value eq 'GRD')"
)

params = {
    "$filter": odata_filter,
    "$top": "100",
    "$select": "Id,Name,ContentDate,GeoFootprint",
}
j = requests.get(CATALOGUE, params=params, timeout=60).json()
items = j.get("value", [])
items = sorted(items, key=lambda x: x["ContentDate"]["Start"])
print(f"Found: {len(items)} Sentinel-1 GRD scenes\n")

for p in items:
    print(f"  {p['Name']}  |  {p['ContentDate']['Start'][:19]}")

# Download first 5 via zipper endpoint (handles auth correctly)
os.makedirs("data/sentinel1", exist_ok=True)
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}"})

print(f"\nDownloading first 5 scenes...")
for p in items[:5]:
    pid = p["Id"]
    name = p["Name"]
    out = os.path.join("data/sentinel1", f"{name}.zip")
    if os.path.exists(out):
        print(f"  Already exists: {name}")
        continue
    url = f"{DOWNLOAD}({pid})/$value"
    with session.get(url, stream=True, timeout=600, allow_redirects=False) as r:
        # Follow redirect manually with token
        if r.status_code in (301, 302, 303, 307):
            redirect_url = r.headers["Location"]
            r = session.get(redirect_url, stream=True, timeout=600)
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        downloaded = 0
        with open(out, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
        size_mb = downloaded / (1024 * 1024)
        print(f"  Downloaded: {name} ({size_mb:.0f} MB)")

Found: 27 Sentinel-1 GRD scenes

  S1B_IW_GRDH_1SDV_20200611T043639_20200611T043704_021981_029B85_067B.SAFE  |  2020-06-11T04:36:39
  S1B_IW_GRDH_1SDV_20200611T043639_20200611T043704_021981_029B85_07F2.SAFE  |  2020-06-11T04:36:39
  S1A_IW_GRDH_1SDV_20200612T042904_20200612T042929_032979_03D1E7_D8AA.SAFE  |  2020-06-12T04:29:04
  S1A_IW_GRDH_1SDV_20200612T042904_20200612T042929_032979_03D1E7_9D9D.SAFE  |  2020-06-12T04:29:04
  S1A_IW_GRDH_1SDV_20200613T161808_20200613T161833_033001_03D293_419C.SAFE  |  2020-06-13T16:18:08
  S1A_IW_GRDH_1SDV_20200613T161808_20200613T161833_033001_03D293_0A33.SAFE  |  2020-06-13T16:18:08
  S1B_IW_GRDH_1SDV_20200614T160915_20200614T160940_022032_029D01_DFB8.SAFE  |  2020-06-14T16:09:15
  S1B_IW_GRDH_1SDV_20200614T160915_20200614T160940_022032_029D01_E5C9.SAFE  |  2020-06-14T16:09:15
  S1A_IW_GRDH_1SDV_20200617T043721_20200617T043746_033052_03D426_1D3A.SAFE  |  2020-06-17T04:37:21
  S1A_IW_GRDH_1SDV_20200617T043721_20200617T043746_033052_03D426_5C6C.SAFE  

ChunkedEncodingError: ('Connection broken: IncompleteRead(939141855 bytes read, 836675310 more expected)', IncompleteRead(939141855 bytes read, 836675310 more expected))

## Обробка Sentinel-1 для маски води (повінь Буковина, червень 2020)

Пайплайн обробки (SNAP GPT):
1. **ThermalNoiseRemoval** — видалення теплових шумів (VV)
2. **Apply-Orbit-File** — корекція орбітальних даних
3. **Remove-GRD-Border-Noise** — видалення межового шуму
4. **Calibration** — радіометрична калібрація → Sigma0
5. **Speckle-Filter** (Lee Sigma 3x3) — зменшення спеклів
6. **Terrain-Correction** — геокорекція рельєфу (SRTM, EPSG:4326)
7. **BandMaths** — бінаризація: `Sigma0_VV < 0.017` → water_mask (1=вода, 0=суша)

In [ ]:
import subprocess
import glob
import os

SNAP_GPT = "/opt/snap/bin/gpt"
GRAPH_XML = "/work/flood_preprocess.xml"
INPUT_DIR = "/work/data/sentinel1"
OUTPUT_DIR = "/work/data/water_masks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Find all downloaded .zip files
zips = sorted(glob.glob(os.path.join(INPUT_DIR, "*.zip")))
print(f"Found {len(zips)} Sentinel-1 scenes to process\n")

for zf in zips:
    name = os.path.basename(zf).replace(".SAFE.zip", "")
    output_tif = os.path.join(OUTPUT_DIR, f"{name}_water_mask.tif")

    if os.path.exists(output_tif):
        print(f"  Already processed: {name}")
        continue

    print(f"Processing: {name} ...")
    cmd = [
        SNAP_GPT, GRAPH_XML,
        f"-Pinput_file={zf}",
        f"-Poutput_file={output_tif}",
        "-q", "4",  # 4 threads
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)

    if result.returncode == 0:
        size_mb = os.path.getsize(output_tif) / (1024 * 1024)
        print(f"  Done: {output_tif} ({size_mb:.1f} MB)")
    else:
        print(f"  FAILED: {name}")
        print(f"  stderr: {result.stderr[-500:]}")

print("\nAll processing complete!")
print("Output files:")
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.tif"))):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import glob
import os

OUTPUT_DIR = "/work/data/water_masks"
tifs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.tif")))

if not tifs:
    print("No water masks found yet. Run the processing cell first.")
else:
    # Load basin boundary for overlay
    basin = gpd.read_file("/work/basin_bc.geojson").to_crs(4326)

    fig, axes = plt.subplots(1, len(tifs), figsize=(8 * len(tifs), 8))
    if len(tifs) == 1:
        axes = [axes]

    for ax, tif_path in zip(axes, tifs):
        with rasterio.open(tif_path) as src:
            water = src.read(1)
            extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

        # Count water pixels
        total_px = np.count_nonzero(water != 255)  # exclude nodata
        water_px = np.count_nonzero(water == 1)
        water_pct = (water_px / total_px * 100) if total_px > 0 else 0
        water_km2 = water_px * 10 * 10 / 1e6  # 10m pixel

        # Display
        ax.imshow(water, extent=extent, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
        basin.boundary.plot(ax=ax, color="red", linewidth=1.5)

        name = os.path.basename(tif_path).split("_202")[0].split("_")[-1]
        date_str = os.path.basename(tif_path).split("_")[4][:8]
        date_fmt = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"

        ax.set_title(f"{date_fmt}\nWater: {water_pct:.1f}% ({water_km2:.1f} km²)", fontsize=14)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    plt.suptitle("Flood Water Mask — Bukovyna, June 2020", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig("/work/data/water_masks/flood_map.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: /work/data/water_masks/flood_map.png")